# 💎 Mirror Nexus Bot — Heroku Deployment

**Developer:** [@Venuboyy](https://t.me/Venuboyy)

This notebook will deploy your Mirror Nexus Bot to Heroku step by step.

---

## Prerequisites
- GitHub repository with your bot code
- Heroku account (free/eco tier works)
- MongoDB Atlas URI
- Bot Token from [@BotFather](https://t.me/BotFather)
- API ID & Hash from [my.telegram.org](https://my.telegram.org)

---

## Step 1: Install Heroku CLI

In [ ]:
# Install Heroku CLI
!curl https://cli-assets.heroku.com/install.sh | sh
!heroku --version

## Step 2: Login to Heroku

You need your Heroku API key. Get it from: https://dashboard.heroku.com/account

Scroll down to **API Key** → click **Reveal** → copy it.

In [ ]:
import os
from getpass import getpass

# Enter your Heroku API Key (hidden input)
HEROKU_API_KEY = getpass("Enter your Heroku API Key: ")
os.environ["HEROKU_API_KEY"] = HEROKU_API_KEY

# Enter your Heroku email
HEROKU_EMAIL = input("Enter your Heroku Email: ")

# Login using API key
!echo {HEROKU_API_KEY} | heroku auth:token
print("\n✅ Heroku authentication configured!")

## Step 3: Clone Your Repository

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⚠️ EDIT THIS: Your GitHub repository URL
# ═══════════════════════════════════════════════════════════════
GITHUB_REPO = "https://github.com/YOUR_USERNAME/NexusMLTB.git"  # ← CHANGE THIS
BRANCH = "main"  # or "master"

# Clone the repo
!rm -rf /content/NexusMLTB
!git clone -b {BRANCH} {GITHUB_REPO} /content/NexusMLTB
%cd /content/NexusMLTB
print("\n✅ Repository cloned!")

## Step 4: Configure Your Bot

Fill in ALL required values below. Optional ones can be left empty.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔧 BOT CONFIGURATION — Fill in your values
# ═══════════════════════════════════════════════════════════════

# ── REQUIRED ──────────────────────────────────────────────────
HEROKU_APP_NAME = ""        # Your Heroku app name (must be unique)
BOT_TOKEN       = ""        # From @BotFather
API_ID          = ""        # From my.telegram.org
API_HASH        = ""        # From my.telegram.org
OWNER_ID        = ""        # Your Telegram user ID
MONGO_URI       = ""        # MongoDB Atlas connection URI

# ── OPTIONAL ─────────────────────────────────────────────────
SUDO_USERS           = ""   # Comma-separated user IDs
FORCE_SUB_CHANNEL_1  = ""   # Channel ID or username
FORCE_SUB_CHANNEL_2  = ""   # Channel ID or username
DB_NAME              = "mirrornexus"
DOWNLOAD_DIR         = "/tmp/downloads"
MAX_SPLIT_SIZE       = "2000000000"
WORKERS              = "500"
STICKER_ID           = ""   # Custom sticker file_id
WELCOME_IMG_API      = ""   # Welcome image API URL
MASSAGE_IMG_URL      = ""   # Fallback welcome image URL
ARIA2C_SECRET        = ""   # Aria2 RPC secret
GDRIVE_FOLDER_ID     = ""   # Google Drive folder ID

print("✅ Configuration set!")

## Step 5: Create Heroku App

Choose your deployment method:
- **Method A: Docker (Recommended)** — Uses Dockerfile, all system deps included
- **Method B: Buildpack** — Uses Procfile + Aptfile, simpler but less flexible

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Choose deployment method: "docker" or "buildpack"
# ═══════════════════════════════════════════════════════════════
DEPLOY_METHOD = "docker"  # ← "docker" (recommended) or "buildpack"

# Create the Heroku app
!heroku create {HEROKU_APP_NAME} --region us
print(f"\n✅ App '{HEROKU_APP_NAME}' created!")

if DEPLOY_METHOD == "docker":
    # Set stack to container for Docker deployment
    !heroku stack:set container -a {HEROKU_APP_NAME}
    print("📦 Stack set to container (Docker)")
else:
    # Add buildpacks for non-Docker deployment
    !heroku buildpacks:add --index 1 heroku-community/apt -a {HEROKU_APP_NAME}
    !heroku buildpacks:add --index 2 heroku/python -a {HEROKU_APP_NAME}
    print("📦 Buildpacks configured (apt + python)")

## Step 6: Set Environment Variables on Heroku

In [ ]:
# Set all config vars on Heroku
config_vars = {
    "BOT_TOKEN":            BOT_TOKEN,
    "API_ID":               API_ID,
    "API_HASH":             API_HASH,
    "OWNER_ID":             OWNER_ID,
    "MONGO_URI":            MONGO_URI,
    "HEROKU_APP_NAME":      HEROKU_APP_NAME,
    "DB_NAME":              DB_NAME,
    "DOWNLOAD_DIR":         DOWNLOAD_DIR,
    "MAX_SPLIT_SIZE":       MAX_SPLIT_SIZE,
    "WORKERS":              WORKERS,
}

# Add optional vars only if set
optional = {
    "SUDO_USERS":           SUDO_USERS,
    "FORCE_SUB_CHANNEL_1":  FORCE_SUB_CHANNEL_1,
    "FORCE_SUB_CHANNEL_2":  FORCE_SUB_CHANNEL_2,
    "STICKER_ID":           STICKER_ID,
    "WELCOME_IMG_API":      WELCOME_IMG_API,
    "MASSAGE_IMG_URL":      MASSAGE_IMG_URL,
    "ARIA2C_SECRET":        ARIA2C_SECRET,
    "GDRIVE_FOLDER_ID":     GDRIVE_FOLDER_ID,
}

for k, v in optional.items():
    if v:
        config_vars[k] = v

# Build the heroku config:set command
config_string = " ".join(f'{k}="{v}"' for k, v in config_vars.items() if v)
!heroku config:set {config_string} -a {HEROKU_APP_NAME}

print("\n✅ All environment variables configured!")
print(f"📋 Total vars set: {len([v for v in config_vars.values() if v])}")

## Step 7: Deploy to Heroku

In [ ]:
# Initialize git and push to Heroku
%cd /content/NexusMLTB

# Add Heroku remote
!heroku git:remote -a {HEROKU_APP_NAME}

# Configure git
!git config user.email "{HEROKU_EMAIL}"
!git config user.name "NexusBot"

# Stage all changes
!git add -A
!git commit -m "Deploy Mirror Nexus Bot to Heroku" --allow-empty

# Push to Heroku
!git push heroku {BRANCH}:main -f

print("\n✅ Code pushed to Heroku!")

## Step 8: Scale the Worker Dyno

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IMPORTANT: Scale worker dyno and disable web dyno
# Bot uses 'worker' dyno (long-running process, not a web app)
# ═══════════════════════════════════════════════════════════════

# Turn off web dyno (if exists) and enable worker dyno
!heroku ps:scale web=0 worker=1 -a {HEROKU_APP_NAME}

print("\n✅ Worker dyno scaled!")
print("   web=0 (disabled)")
print("   worker=1 (running)")

## Step 9: Check Status & Logs

In [ ]:
# Check dyno status
print("═" * 50)
print("📊 DYNO STATUS")
print("═" * 50)
!heroku ps -a {HEROKU_APP_NAME}

print("\n" + "═" * 50)
print("📋 CONFIG VARS")
print("═" * 50)
!heroku config -a {HEROKU_APP_NAME}

In [ ]:
# View recent logs (last 100 lines)
!heroku logs --tail -n 100 -a {HEROKU_APP_NAME}

---

## 🔧 Utility Commands

Run these as needed to manage your deployment.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔄 RESTART BOT
# ═══════════════════════════════════════════════════════════════
!heroku ps:restart worker -a {HEROKU_APP_NAME}
print("✅ Bot restarted!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🛑 STOP BOT
# ═══════════════════════════════════════════════════════════════
!heroku ps:scale worker=0 -a {HEROKU_APP_NAME}
print("🛑 Bot stopped!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ▶️ START BOT
# ═══════════════════════════════════════════════════════════════
!heroku ps:scale worker=1 -a {HEROKU_APP_NAME}
print("▶️ Bot started!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔄 UPDATE & REDEPLOY (pull latest code + push)
# ═══════════════════════════════════════════════════════════════
%cd /content/NexusMLTB
!git pull origin {BRANCH}
!git add -A
!git commit -m "Update deployment" --allow-empty
!git push heroku {BRANCH}:main -f
!heroku ps:restart worker -a {HEROKU_APP_NAME}
print("\n✅ Updated and redeployed!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📝 UPDATE A SINGLE ENV VAR
# ═══════════════════════════════════════════════════════════════
VAR_NAME  = "BOT_TOKEN"     # ← Change to the var you want to update
VAR_VALUE = ""              # ← New value

!heroku config:set {VAR_NAME}="{VAR_VALUE}" -a {HEROKU_APP_NAME}
print(f"✅ {VAR_NAME} updated!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🗑️ DELETE HEROKU APP (DANGER!)
# ═══════════════════════════════════════════════════════════════
# Uncomment the line below to delete the app
# !heroku apps:destroy {HEROKU_APP_NAME} --confirm {HEROKU_APP_NAME}
# print("🗑️ App deleted!")

---

## ✅ Deployment Complete!

Your Mirror Nexus Bot is now running on Heroku! 🚀

### Troubleshooting

| Issue | Solution |
|-------|----------|
| Bot not responding | Check logs: `heroku logs --tail` |
| R10 Boot Timeout | Make sure `PORT` binding works |
| H14 No Web Dynos | Scale worker: `heroku ps:scale web=0 worker=1` |
| Slug too large | Use Docker method instead of buildpack |
| Crash loop | Check env vars are correctly set |

### Links
- 📊 [Heroku Dashboard](https://dashboard.heroku.com)
- 💬 [Developer: @Venuboyy](https://t.me/Venuboyy)
- 📢 [Channel: @zerodev2](https://t.me/zerodev2)